# 04 · Train the adapter

**Examples → prediction → loss → adapter update**

Use the practice data from notebook 02 and the CPU adapter setup from notebook 03. We train for **40 updates**. The large Lightning models keep serving as before.

**Who does what:** Nano coordinates the tools. PyTorch trains the tiny CPU adapter. Lightning is not called or trained in this lesson.

**CPU training is for demonstration purposes only.** These are real optimizer updates on a tiny practice model, not Lightning fine-tuning. To train an actual Lightning adapter on the DGX GPU, follow [train_lightning.py](scripts/train_lightning.py) and the optional instructions below.


###  optimization and validation loss

PyTorch's [optimization tutorial](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) connects loss, backpropagation, and optimizer updates. The [PEFT guide](https://huggingface.co/docs/peft/main/en/quicktour) explains adapter parameters. Match the [tested versions](README.md#concept-sdks-and-documentation).

finish manual 02 and 03, then load → batch → baseline → train. This runs 40 real CPU updates and saves `cpu/student-adapter` and measured metrics. No Lightning generation is needed in the manual exercise.

**Edit the actual implementation below:** `encode`, `batch`, `evaluate`, and the optimizer loop are executable cells. The [04 reference lesson](agent/lessons/04-train-adapter.json) is reserved for the fixed agent handoff. The code cells below execute these operations directly; [the execution guide](README.md#how-the-notebook-helpers-execute) distinguishes manual experiments from fixed agent recipes.

**Read the result carefully:** the Transformers model computes the causal-language-model loss from masked labels. Context/padding are ignored; gradients update the adapter. Lower loss is a diagnostic, not proof of useful generated answers. Notebook 06 tests those answers.

The agent uses a workshop-authored training skill. See [Skills and training](README.md#skills-and-training) for the distinction between these instructions, NeMo libraries, and the separate NeMo Skills pipeline.

**Optional NVIDIA training reference:** [AutoModel 0.4.0 / 26.04 ChatDataset](https://docs.nvidia.com/nemo/automodel/v0.4/nemo-automodel/nemo_automodel/components/datasets/llm/chat_dataset) documents the loader used by [train_lightning.py](scripts/train_lightning.py). That separate script checks pre-shifted label alignment before loading weights; the CPU lesson lets Transformers shift labels internally. Find `nemo-automodel-recipe-development` in [NVIDIA’s skills catalog](https://build.nvidia.com/skills); it is an optional review reference, not a skill installed or executed by this notebook.


**Manual experiment versus agent reference:** the executable cells below are yours to edit. The OpenClaw section at the end runs the fixed workshop recipe in `agent/lessons/`; it does not ingest your edited cells or reuse manual artifacts. Setup, displays, artifact bookkeeping, and terminal/process transport remain helpers.

### Start here
Run the executable cells in order. Edit the visible requests, rules, and experiment parameters to change your manual run. The artifact helper only copies prerequisites and records output paths. Starting again creates a fresh output folder.

In [1]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import table, bars, loss_chart, show_skill_library
from course.agent.notebook_support import NotebookRun
from course.common import chat, request, rows, BASE_URL
import os, time

lesson = NotebookRun('04-train-adapter')
OUTPUT = lesson.work
LIVE = True


Fresh manual artifacts: /home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/outputs/previews/04-train-adapter/0191ee4cb9014cbda61ed8b067fb3513/work


### 1 · Load the starting point

The saved base and adapter recipe become a fresh training model. The setup prints the exact input folders; complete the manual sections of notebooks 02 and 03 first.

In [2]:
import torch, random
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
PROFILE = 'cpu'  # Must match notebook 03; Lightning requires dedicated GPU memory.
BUILD = OUTPUT/PROFILE
train_rows = [json.loads(line) for line in (OUTPUT/'student-data/train.jsonl').read_text().splitlines()]
val_rows = [json.loads(line) for line in (OUTPUT/'student-data/validation.jsonl').read_text().splitlines()]
torch.manual_seed(17); random.seed(17); torch.set_num_threads(2)
base = AutoModelForCausalLM.from_pretrained(BUILD/'base',local_files_only=True)
tokenizer = None
model = get_peft_model(base,LoraConfig.from_pretrained(BUILD/'adapter-config'))
device = next(model.parameters()).device
print('Training on',device,'profile',PROFILE)

/home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 16/16 [00:00<00:00, 36832.53it/s]

Training on cpu profile cpu


In [3]:
print("Training device:", device)

Training device: cpu


### 2 · See a training batch

Context and padding are ignored by the loss; answer tokens are scored. This practice model uses byte IDs, unlike Lightning’s tokenizer.

In [4]:
def encode(row):
    messages = row['messages']
    prefix = 'Observation: '+messages[1]['content']+'\nAnswer: '
    answer = messages[-1]['content']
    prompt_ids = [b+2 for b in prefix.encode()]
    answer_ids = [b+2 for b in answer.encode()]+[1]
    return prompt_ids+answer_ids, [-100]*len(prompt_ids)+answer_ids

def batch(rows):
    encoded = [encode(row) for row in rows]
    width = max(len(ids) for ids,labels in encoded)
    pad = 0
    ids = torch.full((len(rows),width),pad,dtype=torch.long,device=device)
    labels = torch.full_like(ids,-100)
    mask = torch.zeros_like(ids)
    for i,(tokens,targets) in enumerate(encoded):
        ids[i,:len(tokens)] = torch.tensor(tokens,device=device)
        labels[i,:len(tokens)] = torch.tensor(targets,device=device)
        mask[i,:len(tokens)] = 1
    return {'input_ids':ids,'attention_mask':mask,'labels':labels}
example_batch = batch(train_rows[:2])
print({key:tuple(value.shape) for key,value in example_batch.items()})
print('Supervised tokens:',(example_batch['labels']!=-100).sum().item())

{'input_ids': (2, 269), 'attention_mask': (2, 269), 'labels': (2, 269)}
Supervised tokens: 223


In [5]:
batch_view = example_batch
table(["Batch component", "Shape"], [(k, tuple(v.shape)) for k,v in batch_view.items()])
print("Answer tokens scored:", int((batch_view["labels"] != -100).sum()))

Batch component,Shape
input_ids,"(2, 269)"
attention_mask,"(2, 269)"
labels,"(2, 269)"


Answer tokens scored: 223


### 3 · Measure before training

Validation examples stay outside training. First measure how well the untrained adapter predicts their answers.

In [6]:
LEARNING_RATE = 3e-3  # Change this and rerun baseline/train.
parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(parameters,lr=LEARNING_RATE)
# Keep one frozen tensor as a check without duplicating the full model.
frozen_name,frozen_parameter = next((name,p) for name,p in model.named_parameters() if not p.requires_grad)
frozen_before = frozen_parameter.detach().flatten()[:128].cpu().clone()
@torch.no_grad()
def evaluate():
    model.eval()
    values = [model(**batch([row])).loss.item() for row in val_rows]
    return sum(values)/len(values)
before = evaluate()
print('Validation loss before:',before)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Validation loss before: 5.5550217628479


In [7]:
print("Validation loss before:", before)

Validation loss before: 5.5550217628479


### 4 · Train and watch the loss

Run 40 real CPU optimizer updates, then save the adapter. The curve is measured training loss; the bars compare validation loss before and after.

Lower loss does not prove good generated answers. The existing lesson checks a sample from one frozen tensor; it is not a full-weight audit.

In [8]:
STEPS = 40  # Smoke budget; tune based on evaluation.
ACCUMULATION = 2
history = []
model.train()
for step in range(STEPS):
    optimizer.zero_grad()
    losses = []
    for microstep in range(ACCUMULATION):
        selected = [random.choice(train_rows)]
        output = model(**batch(selected))
        loss = output.loss
        assert torch.isfinite(loss)
        (loss/ACCUMULATION).backward()
        losses.append(loss.item())
    norm = torch.nn.utils.clip_grad_norm_(parameters,1.0)
    assert torch.isfinite(norm)
    optimizer.step()
    history.append({'step':step,'loss':sum(losses)/len(losses),'gradient_norm':norm.item()})
    if step%10==0: print(history[-1])
after = evaluate()
assert torch.equal(frozen_before,frozen_parameter.detach().flatten()[:128].cpu())
print('Validation loss:',before,'→',after,'| frozen base check passed')

{'step': 0, 'loss': 5.5580644607543945, 'gradient_norm': 0.1976289302110672}
{'step': 10, 'loss': 5.349648475646973, 'gradient_norm': 0.2558940351009369}
{'step': 20, 'loss': 5.295510292053223, 'gradient_norm': 0.28917959332466125}
{'step': 30, 'loss': 5.257247447967529, 'gradient_norm': 0.17643079161643982}
Validation loss: 5.5550217628479 → 5.201232274373372 | frozen base check passed


In [9]:
candidate = BUILD/'student-adapter'
model.save_pretrained(candidate)
if tokenizer is not None: tokenizer.save_pretrained(candidate/'tokenizer')
metrics = {'profile':PROFILE,'validation_before':before,'validation_after':after,
           'steps':STEPS,'history':history,'base_frozen':True}
(BUILD/'training-metrics.json').write_text(json.dumps(metrics,indent=2))
print('Your new adapter:',candidate)
for file in candidate.iterdir(): print(file.name,file.stat().st_size)

Your new adapter: /home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/outputs/previews/04-train-adapter/0191ee4cb9014cbda61ed8b067fb3513/work/cpu/student-adapter
README.md 5480
adapter_config.json 1251
adapter_model.safetensors 25568


In [10]:
metrics = lesson.read("cpu/training-metrics.json")
loss_chart(metrics["history"])
bars({"Validation before": metrics["validation_before"], "Validation after": metrics["validation_after"]})

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [11]:
assert len(history) == STEPS and metrics['base_frozen']
assert torch.equal(frozen_before, frozen_parameter.detach().flatten()[:128].cpu())
manual_result = lesson.finish({'steps': STEPS, 'loss_before': before, 'loss_after': after,
                               'frozen_sample_unchanged': True})
display(JSON(manual_result, expanded=False))

<IPython.core.display.JSON object>

### Optional · Train actual Lightning on the DGX GPU

Follow the repository's [train_lightning.py](scripts/train_lightning.py) reproduction recipe. Here **Lightning means NVIDIA Nemotron 3.5 Lightning**, not the PyTorch Lightning framework. The recipe uses Transformers, PEFT and NeMo AutoModel's `ChatDataset`, with an explicit PyTorch CUDA training loop.

1. Use the separate `nemo-automodel:26.04` training environment required by the script, with CUDA-enabled PyTorch, Transformers, PEFT and NeMo AutoModel available. The course's CPU environment is not that environment. Reserve GPU memory for model weights, activations, optimizer state and validation; do not assume free capacity alongside the serving models. This script loads onto one CUDA device; it does not configure distributed training.
2. Provide a local [NVIDIA Lightning BF16 checkpoint](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16), and run from the repository root inside the training environment. Set `LIGHTNING_BASE` to that snapshot and `LIGHTNING_OUTPUT` to a new output directory. The script refuses to overwrite an existing directory. Review the [NVIDIA model card](https://build.nvidia.com/nvidia/nemotron-3.5-lightning-30b-a3b/modelcard) for the model's requirements.
3. Check the dataset/tokenizer contract before loading weights. The recipe checks that supervised labels are already shifted; its loss must not shift them a second time. See NVIDIA's [ChatDataset reference](https://docs.nvidia.com/nemo/automodel/v0.4/nemo-automodel/nemo_automodel/components/datasets/llm/chat_dataset).

```bash
# Terminal in the separate GPU training environment, from the repository root.
export LIGHTNING_BASE=/path/to/local/Lightning-BF16-snapshot
export LIGHTNING_OUTPUT=/path/to/new/lightning-adapter-run
python course/scripts/train_lightning.py \
  --base "$LIGHTNING_BASE" \
  --train train/data/jisr/train.jsonl \
  --validation train/data/jisr/val.jsonl \
  --output "$LIGHTNING_OUTPUT" --check-data-only
```

After that check passes, explicitly launch a small GPU reproduction run:

```bash
python course/scripts/train_lightning.py \
  --base "$LIGHTNING_BASE" \
  --train train/data/jisr/train.jsonl \
  --validation train/data/jisr/val.jsonl \
  --output "$LIGHTNING_OUTPUT" \
  --steps 20 --accumulation 8 --lr 2e-4 --seed 1111
```

The recipe attaches rank-32 LoRA to Lightning's configured projection layers, freezes the base, and saves adapter weights/configuration, `training.jsonl`, `tokenizer/` and `run.json`. Inspect finite losses, validation loss and generated answers. Its first-12-example validation and three generations are diagnostics; use the held-out task checks from notebook 06 before drawing quality conclusions. Twenty updates are a smoke experiment, not an established training budget.

This optional recipe has not been GPU-validated as part of this documentation update. It does not run with notebook Run All or replace the deployed specialist. The tiny CPU adapter cannot be reused for Lightning. For serving a compatible trained adapter, continue to the optional NIM instructions in [notebook 05](05-attach-and-serve.ipynb). For reward-driven agent improvement, continue to the full Gym/RL exercise in [notebook 11](11-trace-and-improve.ipynb); SFT and RL are separate training stages.

### Grow the skill library
The library now includes this lesson’s skill, **workshop-train-adapter**. Open the actual instructions below.

**`inspect-inputs` → `train-adapter` → `check-training`**

In [12]:
show_skill_library('04-train-adapter')

Notebook,Skill
1,workshop-model-inventory
2,workshop-training-data
3,workshop-build-lora
4,workshop-train-adapter


### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-train-adapter/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-train-adapter/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

Complete these earlier **OpenClaw demos** first: `02-training-data`, `03-build-lora`. This run copies their latest verified agent artifacts; its inspect action reports the inputs.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [13]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('04-train-adapter')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-train-adapter/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-inputs', 'train-adapter', 'check-training']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

Local run directory: /home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/outputs/agent-demos/04-train-adapter/openclaw-fccca81cce3e4d56810c9d49f71ae8b0
OpenClaw sandbox workspace: /home/sandbox/course-runs/openclaw-fccca81cce3e4d56810c9d49f71ae8b0/workspace
Installed skill: /home/sandbox/course-runs/openclaw-fccca81cce3e4d56810c9d49f71ae8b0/workspace/skills/workshop-train-adapter/SKILL.md
Python helper: /home/sandbox/course-runs/openclaw-fccca81cce3e4d56810c9d49f71ae8b0/chapter_tool.py

The agent will run these commands individually:
python3 /home/sandbox/course-runs/openclaw-fccca81cce3e4d56810c9d49f71ae8b0/chapter_tool.py inspect-inputs
python3 /home/sandbox/course-runs/openclaw-fccca81cce3e4d56810c9d49f71ae8b0/chapter_tool.py train-adapter
python3 /home/sandbox/course-runs/openclaw-fccca81cce3e4d56810c9d49f71ae8b0/chapter_tool.py check-training


### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano coordinates the tools. PyTorch trains the tiny CPU adapter. Lightning is not called or trained in this lesson.

In [14]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

Requested an OpenClaw terminal on the workshop computer’s desktop. When its TUI is ready, paste the task below.
If no window appears, in a Jupyter terminal run: bash /home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/outputs/agent-demos/04-train-adapter/openclaw-fccca81cce3e4d56810c9d49f71ae8b0/open-terminal.sh

PASTE THIS INTO OPENCLAW:

Use the workshop-train-adapter skill. First read /home/sandbox/course-runs/openclaw-fccca81cce3e4d56810c9d49f71ae8b0/workspace/skills/workshop-train-adapter/SKILL.md. The tool path is /home/sandbox/course-runs/openclaw-fccca81cce3e4d56810c9d49f71ae8b0/chapter_tool.py. Task: Train the small CPU adapter for 40 steps, save it, and compare loss before and after. The fixed CPU recipe runs on the host. It does not train or restart the serving models. Run inspect-inputs, then train-adapter, then check-training individually with exec elevated=false, yieldMs=10000, timeoutSeconds=180. If a command is still running, poll with pro

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** 40 training steps completed; adapter saved; frozen-weight check passed.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** The adapter has learned from examples. Next, reload it and inspect what actually changed.

Exit the TUI with `/exit` when done.